In [ ]:
# -*- coding: utf-8 -*-
# Execution environment pie — aligned with Observation 3
# Cohort: CI-executing adopters (RQ1_F1 == True and RQ1_F2 in {Fully_Auto, CI_Only})
# One primary environment per repo (use ..._MR if present). Includes “Unknown”.
# Sums to 100% over the CI-executing cohort.

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ========= CONFIG =========
MODE = "legend"  # "legend" or "labels"
WINDOWS_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1")
PRIORITY_FILES = [
    "5.0_Total_Repo_MR.csv", "5.0_Total_Repo.csv",
    "3.2_Total_Repo_MR.csv", "3.2_Total_Repo.csv",
    "Total_Repo_MR.csv", "Total_Repo.csv",
]
FALLBACKS = [
    Path("/mnt/data/5.0_Total_Repo_MR.csv"),
    Path("/mnt/data/5.0_Total_Repo.csv"),
    Path("/mnt/data/3.2_Total_Repo_MR.csv"),
    Path("/mnt/data/3.2_Total_Repo.csv"),
]
SAVE_TO = None  # e.g., r"C:\Android Mobile App\figs\obs3_env_pie.png"
TITLE = "Execution environment among CI-executing adopters (Fully_Auto ∪ CI_Only)"

def choose_dataset_path():
    if WINDOWS_DIR.exists():
        for name in PRIORITY_FILES:
            p = WINDOWS_DIR / name
            if p.exists():
                return p
        csvs = sorted(WINDOWS_DIR.glob("*.csv"))
        if csvs:
            return csvs[0]
    for p in FALLBACKS:
        if p.exists():
            return p
    raise FileNotFoundError("Dataset not found in the Windows path or fallbacks.")

# Map normalized environment labels (single primary per repo)
def normalize_env(value: str) -> str:
    s = ("" if pd.isna(value) else str(value)).strip().lower()
    if s == "" or s == "unknown" or s == "none":
        return "Unknown"
    if "gmd" in s:
        return "Emulator — GMD (managed)"
    if "third" in s:
        return "Third-party device lab"
    if "real" in s:
        return "Real device (self-hosted)"
    if "emulator" in s:
        return "Emulator — Generic/DIY"
    # Fallback for unexpected tokens
    return "Unknown"

# ========= LOAD & COHORT =========
csv_path = choose_dataset_path()
df = pd.read_csv(csv_path, low_memory=False)

# Required columns
for col in ["RQ1_F1", "RQ1_F2"]:
    if col not in df.columns:
        raise KeyError(f"Expected '{col}' in dataset.")

# Prefer MR columns (tie-resolved)
env_col = "execution_environment_MR" if "execution_environment_MR" in df.columns else "execution_environment"
if env_col not in df.columns:
    raise KeyError("Expected execution environment column not found (…_MR or base).")

# CI-executing adopters
cohort = df[(df["RQ1_F1"] == True) & (df["RQ1_F2"].isin(["Fully_Auto", "CI_Only"]))].copy()
N_ci = len(cohort)

# Normalize environment labels (single value per repo)
cohort["env_norm"] = cohort[env_col].map(normalize_env)

# Fixed order to match the paper
order = [
    "Emulator — Generic/DIY",
    "Emulator — GMD (managed)",
    "Third-party device lab",
    "Real device (self-hosted)",
    "Unknown",
]

counts = cohort["env_norm"].value_counts().reindex(order, fill_value=0)
labels = [lab for lab, c in counts.items() if c > 0]
values = [c for c in counts.values if c > 0]

def autopct_counts(total):
    def _fmt(pct):
        # matplotlib passes percentage; we look up index via internal counter
        idx = _fmt.i
        _fmt.i += 1
        n = values[idx]
        return f"{(n/total*100):.1f}%\n(n={n})"
    _fmt.i = 0
    return _fmt

# ========= PLOT =========
title_text = f"{TITLE}\n(N = {N_ci})"

if MODE == "labels":
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_axes([0.06, 0.06, 0.88, 0.88])
    ax.pie(
        values,
        labels=labels,
        autopct=autopct_counts(N_ci),
        startangle=110,
        counterclock=False,
        labeldistance=1.12,
        pctdistance=0.72,
    )
    ax.set_title(title_text, pad=26)
    fig.tight_layout(rect=[0.02, 0.02, 0.98, 0.90])
elif MODE == "legend":
    fig = plt.figure(figsize=(9, 6.8))
    ax = fig.add_subplot(111)
    wedges, _, _ = ax.pie(
        values,
        labels=None,
        autopct=autopct_counts(N_ci),
        startangle=110,
        counterclock=False,
        pctdistance=0.75,
    )
    ax.set_title(title_text, pad=20)
    legend_labels = [f"{lab} — {values[i]} ({values[i]/N_ci*100:.1f}%)" for i, lab in enumerate(labels)]
    ax.legend(
        wedges, legend_labels,
        loc="center left", bbox_to_anchor=(1.0, 0.5),
        borderaxespad=1.0, title="Categories",
    )
    fig.tight_layout(rect=[0.02, 0.02, 0.86, 0.98])
else:
    raise ValueError("MODE must be 'legend' or 'labels'.")

# Helpful footer
fig.text(0.01, 0.01, "Cohort = adopters with CI-executed instrumentation (Fully_Auto ∪ CI_Only).", 
         ha="left", va="bottom", fontsize=9)

if SAVE_TO:
    plt.savefig(SAVE_TO, dpi=220, bbox_inches="tight")
plt.show()

print(f"Dataset: {csv_path}")
print(f"Cohort size (CI-executing): N={N_ci}")
print("Counts:", counts.to_dict())


In [3]:
# -*- coding: utf-8 -*-
# Extract minimum emulator setup parameters for *Emulator-labeled adopters*.
# - Filters main dataset to RQ1_F1 == True AND execution_environment_MR == "Emulator"
# - Scans CI/build files to extract api_level/system_image/abi/device_name
# - Also collects potential API levels from dataset columns (api_XX, api_level_total)

import os
import re
import json
import yaml
import pandas as pd
from pathlib import Path
from typing import Dict, List, Set

# === INPUTS ===
MAIN_CSV   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\5.0_Total_Repo_MR.csv"
YML_DIR    = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\All_Config_Files"
OUTPUT_CSV = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.1_Emulator_Params_By_Repo.csv"

REPO_SEARCH_ROOTS = [
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Cloned_All",
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\Cloned_Sample",
]

# === Emulator presence signals in YAML/scripts ===
EMULATOR_SIGNALS = [
    r'uses:\s*reactivecircus/android-emulator-runner',          # GitHub Actions
    r'(?m)^\s*\S*avdmanager\b',                                 # avdmanager create avd
    r'(?m)^\s*\S*sdkmanager\b[^\n"]*system-images;android-\d+', # sdkmanager system-images
    r'(?m)^\s*\S*emulator\b[^\n]*\s(-avd|@)\S+',                # emulator -avd / @AVD
    r'(?m)^\s*\S*android-wait-for-emulator\b',
    r'(?m)^\s*\S*circle-android\s+wait-for-boot\b',
]

# === Parameter regex ===
API_PATTERNS = [
    r'\bapi[-_ ]?level\s*:\s*([0-9]{2,3})\b',
    r'\bapiLevel\s*:\s*([0-9]{2,3})\b',
    r'system-images;android-([0-9]{2,3})\b',
]
SYSIMG_PATTERNS = [
    r'\btarget\s*:\s*(google_apis(?:_playstore)?)\b',
    r'system-images;android-\d{2,3};([a-z0-9_]+)\b',
    r'\b(system[-_ ]?image(?:source)?)\b\s*:\s*(google_apis(?:_playstore)?|aosp[_-]?\w*)',
]
ABI_PATTERNS = [
    r'\b(?:abi|arch)\s*:\s*(x86_64|x86|arm64[-_]?v8a|armeabi[-_]?v7a)\b',
    r'system-images;android-\d{2,3};[a-z0-9_]+;(x86_64|x86|arm64[-_]?v8a|armeabi[-_]?v7a)\b',
]
DEVICE_NAME_PATTERNS = [
    r'\bdevice\s*:\s*([A-Za-z0-9_ \-]+)\b',
    r'\bprofile\s*:\s*([A-Za-z0-9_ \-]+)\b',
    r'\b--device\s+"?([A-Za-z0-9_ \-]+)"?',
    r'\b(avd[-_ ]?name)\s*:\s*([A-Za-z0-9_ \-]+)\b',
]
PARAM_PATTERNS = {
    "api_level": API_PATTERNS,
    "system_image": SYSIMG_PATTERNS,
    "abi": ABI_PATTERNS,
    "device_name": DEVICE_NAME_PATTERNS,
}

FOLLOWABLE_EXTS = (".sh", ".bash", ".bat", ".cmd", ".ps1", ".json", ".yml", ".yaml")

# === Helpers ===
def read_text(p: Path) -> str:
    for enc in ("utf-8", "latin-1"):
        try:
            return p.read_text(encoding=enc, errors="ignore")
        except Exception:
            pass
    return ""

def lower_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def has_emulator_setup(text: str) -> bool:
    return any(re.search(p, text, flags=re.I) for p in EMULATOR_SIGNALS)

VAR_TOKEN_RE = re.compile(
    r'(?:(?:\$|\$\{)\s*[A-Za-z_][A-Za-z0-9_]*\s*\}?|'           # $VAR or ${VAR}
    r'\${{\s*(?:secrets|env|vars|inputs)\.[^}]+}}|'              # ${{ secrets.KEY }}
    r'%\([A-Za-z_][A-Za-z0-9_]*\)s)'                             # %(VAR)s
)

def extract_params_from_text_with_tokens(raw: str, patterns_by_key: dict):
    out_vals: Dict[str, List[str]] = {k: [] for k in patterns_by_key.keys()}
    out_tokens: Dict[str, List[str]] = {k: [] for k in patterns_by_key.keys()}

    # explicit literals
    for key, patterns in patterns_by_key.items():
        for pat in patterns:
            for m in re.finditer(pat, raw, flags=re.I | re.M):
                val = m.group(1) if (m.lastindex and m.group(1)) else m.group(0)
                val = (val or "").strip().strip('"').strip("'")
                if val and val.lower() not in ("api-level", "avd name"):
                    if val not in out_vals[key]:
                        out_vals[key].append(val)

    # nearby tokens (env/inputs) around param-like words
    for key in patterns_by_key.keys():
        window_re = re.compile(rf'(?i)(?:{key}|api[-_ ]?level|system[-_ ]?image|target|abi|arch|device|profile|avd[-_ ]?name)[^\n]{{0,160}}')
        for w in window_re.finditer(raw):
            seg = raw[w.start():w.end()]
            for tm in VAR_TOKEN_RE.finditer(seg):
                tok = tm.group(0)
                if tok not in out_tokens[key]:
                    out_tokens[key].append(tok)

    return out_vals, out_tokens

RUN_LINE = re.compile(r'(?mi)^\s*(?:run|script|command)\s*:\s*(.+)$')
PATH_REF  = re.compile(r'(?P<path>(?:\.{0,2}/|[A-Za-z]:\\)?[A-Za-z0-9._\-/\\]+(?:' + '|'.join([re.escape(e) for e in FOLLOWABLE_EXTS]) + r'))')

def find_file_refs_from_yaml(content: str) -> List[str]:
    refs: List[str] = []
    for m in RUN_LINE.finditer(content):
        line = m.group(1)
        refs.extend([r for r in re.findall(PATH_REF, line)])
    for ref in re.findall(r'["\']([^"\']+\.(?:json|ya?ml|sh|bat|cmd|ps1))["\']', content, flags=re.I):
        refs.append(ref)
    out, seen = [], set()
    for r in refs:
        r_norm = r.strip().strip('"').strip("'")
        if r_norm not in seen:
            seen.add(r_norm); out.append(r_norm)
    return out

def repo_key_from_full_name(full_name: str) -> str:
    return full_name.lower().replace("/", ".")

def find_repo_files(repo_key: str, roots: List[str], rel_path: str) -> List[Path]:
    cand: List[Path] = []
    rel_norm = rel_path.replace("\\", "/").lstrip("./")
    for root in roots:
        owner_repo = repo_key.split(".", 1)
        variants = []
        if len(owner_repo) == 2:
            variants.append(Path(root) / owner_repo[0] / owner_repo[1] / rel_norm)
        variants.append(Path(root) / repo_key / rel_norm)
        for p in variants:
            if p.exists() and p.is_file():
                cand.append(p)
    return cand

def collect_yaml_index(yaml_dir: Path) -> Dict[str, List[Path]]:
    index: Dict[str, List[Path]] = {}
    ydir = Path(yaml_dir)
    for p in ydir.iterdir():
        if p.is_file() and p.suffix.lower() in (".yml", ".yaml"):
            name = p.name.lower()
            if "__" in name:
                repo_key = name.split("__", 1)[0]
                index.setdefault(repo_key, []).append(p)
    return index

def gather_api_levels_from_dataset(row: pd.Series) -> List[str]:
    """Collect potential API levels from columns like api_29, api_30, ..., plus api_level_total if present."""
    levels: List[str] = []
    for c in row.index:
        m = re.fullmatch(r"api_(\d{2,3})", c, flags=re.I)
        if m:
            val = row[c]
            try:
                if pd.notna(val) and float(val) > 0:
                    levels.append(m.group(1))
            except Exception:
                pass
    # Keep unique, sorted numerically
    levels = sorted(set(levels), key=lambda x: int(x))
    return levels

def main():
    # Load dataset
    df = pd.read_csv(MAIN_CSV)
    df = lower_cols(df)

    required = ["full_name", "rq1_f1", "execution_environment_mr"]
    for col in required:
        if col not in df.columns:
            raise KeyError(f"Required column '{col}' not found in {MAIN_CSV}")

    # Target only: adopters AND emulator-labeled
    df_tgt = df[(df["rq1_f1"] == True) &
                (df["execution_environment_mr"].fillna("").str.lower() == "emulator")].copy()

    # Build YAML index
    yml_dir = Path(YML_DIR)
    if not yml_dir.exists():
        raise FileNotFoundError(f"YAML directory not found: {YML_DIR}")
    yaml_index = collect_yaml_index(yml_dir)

    rows = []
    for _, rec in df_tgt.iterrows():
        full_name = str(rec["full_name"]).strip()
        if not full_name:
            continue
        repo_key = repo_key_from_full_name(full_name)

        acc_vals: Dict[str, List[str]]   = {k: [] for k in PARAM_PATTERNS.keys()}
        acc_tokens: Dict[str, List[str]] = {k: [] for k in PARAM_PATTERNS.keys()}
        provenance: Set[str] = set()

        emulator_keyword_seen = False

        # Read YAMLs for this repo
        for yml in yaml_index.get(repo_key, []):
            ytxt = read_text(yml)
            if not ytxt:
                continue

            if has_emulator_setup(ytxt):
                emulator_keyword_seen = True

            vals, toks = extract_params_from_text_with_tokens(ytxt, PARAM_PATTERNS)
            for k in PARAM_PATTERNS.keys():
                for v in vals[k]:
                    if v not in acc_vals[k]: acc_vals[k].append(v)
                for v in toks[k]:
                    if v not in acc_tokens[k]: acc_tokens[k].append(v)
            provenance.add(str(yml))

            # Follow referenced files to pick up indirect definitions
            for ref in find_file_refs_from_yaml(ytxt):
                for base in REPO_SEARCH_ROOTS:
                    for rp in find_repo_files(repo_key, [base], ref):
                        rtxt = read_text(rp)
                        if not rtxt: continue
                        if rp.suffix.lower() == ".json":
                            try:
                                jtxt = json.dumps(json.loads(rtxt))
                                v2, t2 = extract_params_from_text_with_tokens(jtxt, PARAM_PATTERNS)
                            except Exception:
                                v2, t2 = extract_params_from_text_with_tokens(rtxt, PARAM_PATTERNS)
                        elif rp.suffix.lower() in (".yml", ".yaml"):
                            try:
                                ytxt2 = json.dumps(yaml.safe_load(rtxt), default=str)
                                v2, t2 = extract_params_from_text_with_tokens(ytxt2, PARAM_PATTERNS)
                            except Exception:
                                v2, t2 = extract_params_from_text_with_tokens(rtxt, PARAM_PATTERNS)
                        else:
                            v2, t2 = extract_params_from_text_with_tokens(rtxt, PARAM_PATTERNS)
                        for k in PARAM_PATTERNS.keys():
                            for v in v2[k]:
                                if v not in acc_vals[k]: acc_vals[k].append(v)
                            for v in t2[k]:
                                if v not in acc_tokens[k]: acc_tokens[k].append(v)
                        provenance.add(str(rp))

        # Finalize explicit vs token-based status
        def finalize(values_list: List[str], tokens_list: List[str]):
            if values_list:  return "; ".join(values_list), "explicit"
            if tokens_list:  return "; ".join(tokens_list), "env_or_input"
            return "", "unspecified"

        api_level, api_status       = finalize(acc_vals["api_level"],    acc_tokens["api_level"])
        system_image, sysimg_status = finalize(acc_vals["system_image"], acc_tokens["system_image"])
        abi, abi_status             = finalize(acc_vals["abi"],          acc_tokens["abi"])
        device_name, dev_status     = finalize(acc_vals["device_name"],  acc_tokens["device_name"])

        # Dataset-derived API levels for this repo
        api_levels_ds = gather_api_levels_from_dataset(rec)
        api_level_total = rec["api_level_total"] if "api_level_total" in rec.index else ""

        rows.append({
            "full_name": full_name,
            "emulator_label": "Emulator",
            "adopter": True,
            "emulator_keyword_seen": emulator_keyword_seen,
            "api_level_from_files": api_level,        "api_level_from_files_status": api_status,
            "system_image": system_image,             "system_image_status": sysimg_status,
            "abi": abi,                               "abi_status": abi_status,
            "device_name": device_name,               "device_name_status": dev_status,
            "api_levels_from_dataset": "; ".join(api_levels_ds) if api_levels_ds else "",
            "api_level_total": api_level_total,
            "sources": "; ".join(sorted(provenance)) if provenance else "",
        })

    out_df = pd.DataFrame(rows).sort_values("full_name").reset_index(drop=True)
    Path(os.path.dirname(OUTPUT_CSV)).mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved: {OUTPUT_CSV}  (rows={len(out_df)})")
    if not out_df.empty:
        print(out_df.head(15).to_string(index=False))

if __name__ == "__main__":
    main()


Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.1_Emulator_Params_By_Repo.csv  (rows=333)
                     full_name emulator_label  adopter  emulator_keyword_seen                       api_level_from_files api_level_from_files_status                                                                                                   system_image system_image_status          abi   abi_status  device_name device_name_status api_levels_from_dataset  api_level_total                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [4]:
# -*- coding: utf-8 -*-
"""
Group emulator parameters by repo (full_name) and export a tidy table.

Input:
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.1_Emulator_Params_By_Repo.csv

Output:
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.2_Emulator_Params_Grouped.csv

Behavior:
- Groups by full_name
- Builds 4 columns with sorted, de-duplicated, comma-separated values:
    API_Level, System_Image, ABI, Device_Name
- Uses only *explicit* values per the `*_status` columns
- API_Level merges `api_level_from_files` (explicit only) + `api_levels_from_dataset`
"""

from pathlib import Path
import re
import pandas as pd

# -------- paths --------
BASE = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters")
INPUT_CSV  = BASE / "5.1_Emulator_Params_By_Repo.csv"
OUTPUT_CSV = BASE / "5.2_Emulator_Params_Grouped.csv"

# Optional fallback if running outside Windows (comment out if not needed)
FALLBACK = Path("/mnt/data/5.1_Emulator_Params_By_Repo.csv")
if not INPUT_CSV.exists() and FALLBACK.exists():
    INPUT_CSV = FALLBACK

# -------- helpers --------
SEP_RE = re.compile(r"[;,|]")  # split on semicolon, comma, or pipe

def split_values(s: str):
    """Split a cell into tokens by common separators; trim; drop empties."""
    if pd.isna(s):
        return []
    parts = [p.strip().strip('"').strip("'") for p in SEP_RE.split(str(s))]
    return [p for p in parts if p]

def dedup_preserve_order(items, key=None):
    """De-duplicate while preserving first occurrence order."""
    seen = set()
    out = []
    for x in items:
        k = key(x) if key else x
        if k not in seen:
            seen.add(k); out.append(x)
    return out

def numeric_sortable(x: str):
    """Return (int) for numeric tokens; fallback to large value to push non-numeric to end."""
    m = re.fullmatch(r"\d{1,3}", x.strip())
    return int(m.group(0)) if m else 10**9

def collect_param_from_rows(series_vals: pd.Series, series_status: pd.Series | None, only_explicit=True):
    """Collect explicit values from a pair (values, status)."""
    vals = []
    for v, st in zip(series_vals, (series_status if series_status is not None else [None] * len(series_vals))):
        if only_explicit and series_status is not None and str(st).strip().lower() != "explicit":
            continue
        vals.extend(split_values(v))
    return vals

# -------- load --------
df = pd.read_csv(INPUT_CSV)

# Be flexible with column name casing
cols = {c.lower(): c for c in df.columns}
def c(name):  # map a desired lower-name to actual column if present
    return cols.get(name.lower())

required = ["full_name", "api_level_from_files", "api_level_from_files_status",
            "system_image", "system_image_status", "abi", "abi_status",
            "device_name", "device_name_status", "api_levels_from_dataset"]
missing = [r for r in required if c(r) is None]
if missing:
    # We can still proceed if some optional columns are missing; only 'full_name' is mandatory.
    if c("full_name") is None:
        raise KeyError(f"Missing required column 'full_name' in {INPUT_CSV}")
    # Warn (print) but continue
    print("Warning: missing columns:", missing)

# -------- group & aggregate --------
grouped_rows = []
for full_name, g in df.groupby(df[c("full_name")]):
    # API levels from files (explicit only)
    api_from_files = collect_param_from_rows(
        g[c("api_level_from_files")] if c("api_level_from_files") else pd.Series(dtype=str),
        g[c("api_level_from_files_status")] if c("api_level_from_files_status") else None,
        only_explicit=True
    )
    # API levels from dataset (no status column)
    api_from_ds = []
    if c("api_levels_from_dataset"):
        for cell in g[c("api_levels_from_dataset")]:
            api_from_ds.extend(split_values(cell))

    # Combine & clean API levels (keep only numeric-looking tokens)
    api_all = [t for t in api_from_files + api_from_ds if re.fullmatch(r"\d{1,3}", str(t).strip())]
    api_all = dedup_preserve_order(api_all, key=lambda x: x.strip())
    api_all_sorted = sorted(api_all, key=numeric_sortable)
    api_str = ", ".join(api_all_sorted)

    # System image (explicit only)
    sysimg_vals = collect_param_from_rows(
        g[c("system_image")] if c("system_image") else pd.Series(dtype=str),
        g[c("system_image_status")] if c("system_image_status") else None,
        only_explicit=True
    )
    sysimg_vals = dedup_preserve_order(sysimg_vals, key=lambda x: x.lower())
    sysimg_vals_sorted = sorted(sysimg_vals, key=lambda x: x.lower())
    sysimg_str = ", ".join(sysimg_vals_sorted)

    # ABI (explicit only)
    abi_vals = collect_param_from_rows(
        g[c("abi")] if c("abi") else pd.Series(dtype=str),
        g[c("abi_status")] if c("abi_status") else None,
        only_explicit=True
    )
    abi_vals = dedup_preserve_order(abi_vals, key=lambda x: x.lower())
    abi_vals_sorted = sorted(abi_vals, key=lambda x: x.lower())
    abi_str = ", ".join(abi_vals_sorted)

    # Device name (explicit only)
    dev_vals = collect_param_from_rows(
        g[c("device_name")] if c("device_name") else pd.Series(dtype=str),
        g[c("device_name_status")] if c("device_name_status") else None,
        only_explicit=True
    )
    dev_vals = dedup_preserve_order(dev_vals, key=lambda x: x.lower())
    dev_vals_sorted = sorted(dev_vals, key=lambda x: x.lower())
    dev_str = ", ".join(dev_vals_sorted)

    grouped_rows.append({
        "full_name": full_name,
        "API_Level": api_str,
        "System_Image": sysimg_str,
        "ABI": abi_str,
        "Device_Name": dev_str,
    })

out = pd.DataFrame(grouped_rows).sort_values("full_name").reset_index(drop=True)

# -------- save --------
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_CSV, index=False)

print(f"Read from: {INPUT_CSV}")
print(f"Wrote grouped table: {OUTPUT_CSV}")
print(f"Rows: {len(out)}")
print(out.head(12).to_string(index=False))


Read from: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.1_Emulator_Params_By_Repo.csv
Wrote grouped table: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\Min_Parameters\5.2_Emulator_Params_Grouped.csv
Rows: 333
                     full_name          API_Level System_Image    ABI Device_Name
              4ertuk.audioview                 25                                
               a-mabe.openhiit             34, 35              x86_64 pixel_6_pro
a914-gowtham.compose-ratingbar         19, 22, 30                                
       aakira.expandablelayout                 21                                
  abdelaziz-mahdy.pytorch_lite                 33              x86_64     Nexus 6
             ably.ably-flutter             24, 29                                
               achep.acdisplay                 23                                
      activitywatch.aw-android                     google_apis x86_64 